# OR · 09 Network Optimization


In [ ]:
# ⚙️ Preparación de entorno y rutas
# Si esta celda tarda demasiado o se cuelga:
# 1) Abre la paleta de comandos (Ctrl+Shift+P)
# 2) "Jupyter: Restart Kernel"
# 3) "Run All Above/Below" o ejecuta desde la primera celda

import sys
from pathlib import Path

# Detectar raíz del repo (buscando pyproject.toml o carpeta src)
_candidates = [Path.cwd(), *Path.cwd().parents]
_repo_root = None
for _p in _candidates:
    if (_p / 'pyproject.toml').exists() or (_p / 'src').exists():
        _repo_root = _p
        break
if _repo_root is None:
    _repo_root = Path.cwd()

if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

print(f"✅ Entorno listo. Raíz del repo: {_repo_root}")

## 🎯 Objetivos de Aprendizaje

- Definir qué aprenderá el lector (máx. 5–7 puntos).
- Conectar con el caso de uso del dominio (demanda, logística, IoT).
- Incluir resultados verificables (métricas, validaciones, artefactos generados).

## 1️⃣ Configuración del Entorno

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Google OR-Tools
try:
    from ortools.linear_solver import pywraplp
    ORTOOLS_AVAILABLE = True
    print("✅ Google OR-Tools disponible")
except ImportError:
    ORTOOLS_AVAILABLE = False
    print("⚠️  Google OR-Tools no instalado")
    print("   Para instalar: pip install ortools")

from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

# Rutas
DATA_DIR = Path("../../data/raw")
OUTPUT_DIR = Path("../../data/processed/or09_network_optimization")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Directorio datos: {DATA_DIR.resolve()}")
print(f"📂 Salida: {OUTPUT_DIR.resolve()}")

### 🎯 Qué hace este notebook

Este notebook resuelve el **Facility Location Problem** con optimización multiobjetivo para diseñar una red logística óptima.

**Decisiones a optimizar:**
1. ¿Qué Distribution Centers (DCs) abrir/cerrar?
2. ¿Cómo asignar demanda de clientes a DCs?
3. ¿Cómo balancear costos vs nivel de servicio?

**Enfoque:**
```
Minimizar: Costos fijos (abrir DCs) + Costos transporte (distancia × demanda)
Sujeto a: Capacidad de DCs, Cobertura 100% de demanda, Presupuesto
```

**Técnica avanzada: Pareto Frontier**
- No existe UNA solución óptima
- Exploramos trade-off costo vs servicio
- Generamos frontera de Pareto con múltiples soluciones eficientes

**Caso de uso:** Rediseño de red de distribución nacional con restricción presupuestaria pero manteniendo SLAs de servicio.

## 2️⃣ Cargar y Preparar Datos

In [ ]:
# Cargar locations y orders
df_locations = pd.read_csv(DATA_DIR / "locations.csv")
df_orders = pd.read_csv(DATA_DIR / "orders.csv")

# Filtrar facilities candidatas (Distribution Centers)
df_facilities = df_locations[df_locations['location_type'] == 'Distribution Center'].copy()

# Agregar demanda por destino
demand_by_destination = df_orders.groupby('destination')['quantity'].sum().reset_index()
demand_by_destination.columns = ['location_id', 'total_demand']

# Filtrar solo destinos que son customers/stores
customer_locations = df_locations[df_locations['location_type'].isin(['Store', 'Customer'])].merge(
    demand_by_destination, on='location_id', how='left'
)
customer_locations['total_demand'] = customer_locations['total_demand'].fillna(0)

print("📊 Datos Cargados:")
print(f"   - Facilities candidatas (DCs): {len(df_facilities)}")
print(f"   - Puntos de demanda (Customers/Stores): {len(customer_locations)}")
print(f"   - Demanda total: {customer_locations['total_demand'].sum():,.0f} unidades")

display(df_facilities.head())
display(customer_locations.head())

**Datos utilizados:**

- **locations.csv**: Facilities candidatas (DCs) con capacidades y ubicaciones
- **orders.csv**: Demanda histórica por destino (clientes/stores)

Agregamos demanda por destino y calculamos distancias euclidianas entre facilities y clientes para estimar costos de transporte.

## 3️⃣ Calcular Matriz de Distancias

In [ ]:
def calculate_distance_matrix(facilities_df, customers_df):
    """
    Calcular matriz de distancias euclidianas entre facilities y customers.
    
    Returns:
        DataFrame con distancias (facilities en filas, customers en columnas)
    """
    facilities_coords = facilities_df[['latitude', 'longitude']].values
    customers_coords = customers_df[['latitude', 'longitude']].values
    
    # Distancia euclidiana (aproximación)
    distances = cdist(facilities_coords, customers_coords, metric='euclidean')
    
    # Convertir a km (1 grado ≈ 111 km)
    distances_km = distances * 111
    
    # Crear DataFrame
    distance_matrix = pd.DataFrame(
        distances_km,
        index=facilities_df['location_id'].values,
        columns=customers_df['location_id'].values
    )
    
    return distance_matrix

# Calcular matriz
distance_matrix = calculate_distance_matrix(df_facilities, customer_locations)

print("📏 Matriz de Distancias:")
print(f"   - Shape: {distance_matrix.shape}")
print(f"   - Min distancia: {distance_matrix.min().min():.1f} km")
print(f"   - Max distancia: {distance_matrix.max().max():.1f} km")
print(f"   - Distancia promedio: {distance_matrix.mean().mean():.1f} km")

display(distance_matrix.head())

**¿Por qué matriz de distancias?**

La distancia entre facilities y clientes determina:
- **Costos de transporte**: Mayor distancia = mayor costo
- **Nivel de servicio**: Menor distancia = entregas más rápidas
- **Lead times**: Distancia afecta tiempos de entrega

Usamos distancia euclidiana (lat/lon) como proxy. En producción se usaría:
- Google Maps Distance Matrix API
- Rutas reales de TMS (Transportation Management System)
- Tiempos de tránsito históricos

## 4️⃣ Definir Parámetros del Modelo

In [ ]:
# Parámetros de costos
FIXED_COST_PER_FACILITY = 100000  # Costo fijo anual de operar un DC
TRANSPORT_COST_PER_KM_UNIT = 0.5  # $/km/unidad
MAX_FACILITIES_TO_OPEN = 5  # Restricción presupuestaria

# Capacidades de facilities (simuladas)
np.random.seed(42)
df_facilities['capacity'] = np.random.randint(50000, 200000, len(df_facilities))

# Crear diccionarios para el modelo
facilities = df_facilities['location_id'].tolist()
customers = customer_locations['location_id'].tolist()

demand = dict(zip(customer_locations['location_id'], customer_locations['total_demand']))
capacity = dict(zip(df_facilities['location_id'], df_facilities['capacity']))
fixed_cost = {f: FIXED_COST_PER_FACILITY for f in facilities}

# Costo de transporte: distancia * demanda * costo_unitario
transport_cost = {}
for f in facilities:
    for c in customers:
        transport_cost[f, c] = distance_matrix.loc[f, c] * TRANSPORT_COST_PER_KM_UNIT

print("⚙️ Parámetros del Modelo:")
print(f"   - Costo fijo por DC: ${FIXED_COST_PER_FACILITY:,}")
print(f"   - Costo transporte: ${TRANSPORT_COST_PER_KM_UNIT}/km/unidad")
print(f"   - Max facilities: {MAX_FACILITIES_TO_OPEN}")
print(f"   - Capacidad total: {sum(capacity.values()):,} unidades")
print(f"   - Demanda total: {sum(demand.values()):,} unidades")
print(f"   - Utilización teórica: {sum(demand.values()) / sum(capacity.values()) * 100:.1f}%")

**Parámetros del modelo:**

**Costos:**
- **FIXED_COST**: $100,000/año por operar un DC (alquiler, personal, utilities)
- **TRANSPORT_COST**: $0.50 por km por unidad (combustible, depreciación vehículos)

**Restricciones:**
- **MAX_FACILITIES**: Máximo 5 DCs (restricción presupuestaria)
- **Capacity**: Cada DC tiene capacidad máxima (simulada 50k-200k unidades)

Estos parámetros se calibrarían con datos reales de Finance y Operations.

## 5️⃣ Modelo de Optimización: Minimizar Costo Total

In [ ]:
if not ORTOOLS_AVAILABLE:
    print("⚠️ Google OR-Tools no disponible. Instalar con: pip install ortools")
else:
    print("🚀 Construyendo modelo de optimización...")
    
    # Crear solver
    solver = pywraplp.Solver.CreateSolver('SCIP')
    
    if not solver:
        print("❌ No se pudo crear solver SCIP")
    else:
        # Variables de decisión
        # y[f] = 1 si facility f está abierta, 0 si no
        y = {}
        for f in facilities:
            y[f] = solver.BoolVar(f'y_{f}')
        
        # x[f,c] = fracción de demanda del customer c servida por facility f
        x = {}
        for f in facilities:
            for c in customers:
                x[f, c] = solver.NumVar(0, 1, f'x_{f}_{c}')
        
        # Función objetivo: Minimizar costo total
        objective = solver.Objective()
        
        # Costos fijos
        for f in facilities:
            objective.SetCoefficient(y[f], fixed_cost[f])
        
        # Costos de transporte
        for f in facilities:
            for c in customers:
                objective.SetCoefficient(x[f, c], transport_cost[f, c] * demand[c])
        
        objective.SetMinimization()
        
        # Restricciones
        
        # 1. Cada customer debe ser servido al 100%
        for c in customers:
            constraint = solver.Constraint(1, 1, f'demand_{c}')
            for f in facilities:
                constraint.SetCoefficient(x[f, c], 1)
        
        # 2. Capacidad de facilities
        for f in facilities:
            constraint = solver.Constraint(0, capacity[f], f'capacity_{f}')
            for c in customers:
                constraint.SetCoefficient(x[f, c], demand[c])
        
        # 3. Solo asignar a facilities abiertas
        for f in facilities:
            for c in customers:
                constraint = solver.Constraint(-solver.infinity(), 0, f'open_{f}_{c}')
                constraint.SetCoefficient(x[f, c], 1)
                constraint.SetCoefficient(y[f], -1)
        
        # 4. Limitar número de facilities abiertas
        constraint = solver.Constraint(0, MAX_FACILITIES_TO_OPEN, 'max_facilities')
        for f in facilities:
            constraint.SetCoefficient(y[f], 1)
        
        print(f"\n📊 Modelo construido:")
        print(f"   - Variables: {solver.NumVariables()}")
        print(f"   - Restricciones: {solver.NumConstraints()}")
        
        # Resolver
        print("\n⏳ Resolviendo...")
        status = solver.Solve()
        
        if status == pywraplp.Solver.OPTIMAL:
            print("\n✅ Solución ÓPTIMA encontrada!")
            
            # Facilities abiertas
            open_facilities = [f for f in facilities if y[f].solution_value() > 0.5]
            
            print(f"\n🏢 Facilities Abiertas: {len(open_facilities)}")
            for f in open_facilities:
                print(f"   - {f}")
            
            # Costos
            total_cost = solver.Objective().Value()
            fixed_costs = sum(fixed_cost[f] * y[f].solution_value() for f in facilities)
            transport_costs = sum(
                transport_cost[f, c] * demand[c] * x[f, c].solution_value()
                for f in facilities for c in customers
            )
            
            print(f"\n💰 Costos:")
            print(f"   - Costo total: ${total_cost:,.0f}")
            print(f"   - Costos fijos: ${fixed_costs:,.0f} ({fixed_costs/total_cost*100:.1f}%)")
            print(f"   - Costos transporte: ${transport_costs:,.0f} ({transport_costs/total_cost*100:.1f}%)")
            
            # Guardar asignaciones
            assignments = []
            for f in facilities:
                for c in customers:
                    allocation = x[f, c].solution_value()
                    if allocation > 0.01:  # Solo asignaciones significativas
                        assignments.append({
                            'facility': f,
                            'customer': c,
                            'allocation': allocation,
                            'demand_served': demand[c] * allocation,
                            'distance_km': distance_matrix.loc[f, c],
                            'cost': transport_cost[f, c] * demand[c] * allocation
                        })
            
            df_assignments = pd.DataFrame(assignments)
            
        elif status == pywraplp.Solver.FEASIBLE:
            print("\n⚠️ Solución FACTIBLE (no óptima) encontrada")
        else:
            print(f"\n❌ No se encontró solución. Status: {status}")

**Modelo matemático (MIP - Mixed Integer Programming):**

**Variables de decisión:**
- `y[f]` ∈ {0,1}: Variable binaria - 1 si facility f está abierta, 0 si no
- `x[f,c]` ∈ [0,1]: Variable continua - fracción de demanda del cliente c servida por facility f

**Función objetivo:**
```
Minimize: Σ(fixed_cost[f] × y[f]) + Σ(transport_cost[f,c] × demand[c] × x[f,c])
          ↑ Costos fijos              ↑ Costos variables de transporte
```

**Restricciones:**
1. **Demanda completa**: Σ x[f,c] = 1 para todo c (cada cliente 100% servido)
2. **Capacidad**: Σ demand[c] × x[f,c] ≤ capacity[f] para todo f
3. **Solo asignar si abierta**: x[f,c] ≤ y[f] (no asignar a facilities cerradas)
4. **Presupuesto**: Σ y[f] ≤ MAX_FACILITIES

**Solver:** Google OR-Tools SCIP (optimizador de MIP de alta performance)

## 6️⃣ Análisis de Solución Óptima

In [ ]:
if ORTOOLS_AVAILABLE and status == pywraplp.Solver.OPTIMAL:
    # Utilización de facilities
    facility_utilization = df_assignments.groupby('facility').agg({
        'demand_served': 'sum'
    }).reset_index()
    facility_utilization = facility_utilization.merge(
        df_facilities[['location_id', 'capacity']],
        left_on='facility', right_on='location_id'
    )
    facility_utilization['utilization_pct'] = (
        facility_utilization['demand_served'] / facility_utilization['capacity'] * 100
    )
    
    print("📊 Utilización de Facilities:")
    display(facility_utilization[['facility', 'demand_served', 'capacity', 'utilization_pct']])
    
    # Visualizar utilización
    fig = px.bar(
        facility_utilization,
        x='facility',
        y='utilization_pct',
        title="Utilización de Facilities (%)",
        labels={'utilization_pct': 'Utilización (%)', 'facility': 'Facility'},
        color='utilization_pct',
        color_continuous_scale='RdYlGn_r'
    )
    fig.add_hline(y=85, line_dash="dash", line_color="red", annotation_text="85% (recomendado)")
    fig.show()
    
    # Distribución de distancias de servicio
    fig = px.histogram(
        df_assignments,
        x='distance_km',
        weights='demand_served',
        title="Distribución de Distancias de Servicio (ponderada por demanda)",
        labels={'distance_km': 'Distancia (km)', 'count': 'Demanda Servida'},
        nbins=30
    )
    fig.show()
    
    # Métricas de nivel de servicio
    avg_distance = (df_assignments['distance_km'] * df_assignments['demand_served']).sum() / df_assignments['demand_served'].sum()
    max_distance = df_assignments['distance_km'].max()
    customers_within_100km = (df_assignments['distance_km'] <= 100).sum() / len(df_assignments) * 100
    
    print(f"\n📏 Nivel de Servicio:")
    print(f"   - Distancia promedio: {avg_distance:.1f} km")
    print(f"   - Distancia máxima: {max_distance:.1f} km")
    print(f"   - Customers dentro de 100 km: {customers_within_100km:.1f}%")
else:
    print("⚠️ No hay solución óptima para analizar")

**Análisis de la solución óptima:**

Una vez resuelto el modelo, analizamos:

1. **Facilities abiertas**: Qué DCs permanecen operativos
2. **Utilización**: % de capacidad usada en cada DC (ideal: 75-85%)
3. **Asignaciones**: Qué clientes sirve cada facility
4. **Costos**: Desglose de fijos vs transporte
5. **Nivel de servicio**: Distancia promedio de entrega

**Métricas clave:**
- Costo total optimizado
- Utilización de capacidad (evitar sobreutilización o capacidad ociosa)
- Distribución de distancias de servicio

## 7️⃣ Optimización Multiobjetivo: Pareto Frontier

In [ ]:
def solve_with_service_constraint(max_avg_distance_km: float):
    """
    Resolver modelo con restricción adicional de nivel de servicio.
    
    Args:
        max_avg_distance_km: Distancia promedio máxima permitida
    
    Returns:
        Dict con costo total y distancia promedio
    """
    solver = pywraplp.Solver.CreateSolver('SCIP')
    
    # Variables
    y = {f: solver.BoolVar(f'y_{f}') for f in facilities}
    x = {(f, c): solver.NumVar(0, 1, f'x_{f}_{c}') for f in facilities for c in customers}
    
    # Objetivo: minimizar costo
    objective = solver.Objective()
    for f in facilities:
        objective.SetCoefficient(y[f], fixed_cost[f])
    for f in facilities:
        for c in customers:
            objective.SetCoefficient(x[f, c], transport_cost[f, c] * demand[c])
    objective.SetMinimization()
    
    # Restricciones originales
    for c in customers:
        solver.Add(sum(x[f, c] for f in facilities) == 1)
    
    for f in facilities:
        solver.Add(sum(x[f, c] * demand[c] for c in customers) <= capacity[f])
    
    for f in facilities:
        for c in customers:
            solver.Add(x[f, c] <= y[f])
    
    solver.Add(sum(y[f] for f in facilities) <= MAX_FACILITIES_TO_OPEN)
    
    # Restricción de nivel de servicio (distancia promedio ponderada)
    total_demand = sum(demand.values())
    solver.Add(
        sum(
            distance_matrix.loc[f, c] * demand[c] * x[f, c]
            for f in facilities for c in customers
        ) <= max_avg_distance_km * total_demand
    )
    
    # Resolver
    status = solver.Solve()
    
    if status == pywraplp.Solver.OPTIMAL:
        total_cost = solver.Objective().Value()
        
        # Calcular distancia promedio real
        total_distance_demand = sum(
            distance_matrix.loc[f, c] * demand[c] * x[f, c].solution_value()
            for f in facilities for c in customers
        )
        avg_distance = total_distance_demand / total_demand
        
        num_facilities = sum(y[f].solution_value() for f in facilities)
        
        return {
            'cost': total_cost,
            'avg_distance_km': avg_distance,
            'num_facilities': num_facilities,
            'status': 'optimal'
        }
    else:
        return {
            'cost': None,
            'avg_distance_km': max_avg_distance_km,
            'num_facilities': None,
            'status': 'infeasible'
        }

if ORTOOLS_AVAILABLE:
    print("🔄 Explorando Pareto Frontier (puede tardar 2-3 minutos)...\n")
    
    # Probar diferentes niveles de servicio
    service_levels = [50, 75, 100, 125, 150, 200]
    pareto_results = []
    
    for max_dist in service_levels:
        print(f"   Probando max_distance = {max_dist} km...")
        result = solve_with_service_constraint(max_dist)
        pareto_results.append(result)
    
    df_pareto = pd.DataFrame(pareto_results)
    df_pareto = df_pareto[df_pareto['status'] == 'optimal']  # Solo soluciones óptimas
    
    print("\n✅ Pareto Frontier calculado")
    display(df_pareto)
else:
    print("⚠️ OR-Tools no disponible para análisis Pareto")

**Pareto Frontier - Optimización Multiobjetivo:**

En la práctica, hay **trade-off entre costo y servicio**:
- Más facilities = mejor servicio (menor distancia) pero **mayor costo fijo**
- Menos facilities = menor costo pero **peor servicio** (mayor distancia)

**¿Cómo exploramos este trade-off?**

Resolvemos el modelo múltiples veces con diferentes restricciones de nivel de servicio:
```python
# Ejemplo: forzar distancia promedio ≤ 100 km
solver.Add(avg_distance <= 100)
```

Cada solución óptima es un **punto en la Frontera de Pareto**:
- Puntos donde NO es posible mejorar un objetivo sin empeorar el otro
- Decisión final depende de prioridades estratégicas del negocio

Esto puede tardar 2-3 minutos ya que resolvemos 6 problemas de optimización.

## 8️⃣ Visualizar Pareto Frontier

In [ ]:
if ORTOOLS_AVAILABLE and len(df_pareto) > 0:
    # Pareto Frontier: Costo vs Nivel de Servicio
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df_pareto['avg_distance_km'],
        y=df_pareto['cost'],
        mode='markers+lines',
        marker=dict(size=12, color=df_pareto['num_facilities'], colorscale='Viridis', showscale=True,
                   colorbar=dict(title="# Facilities")),
        line=dict(color='blue', width=2),
        text=[f"Facilities: {int(nf)}" for nf in df_pareto['num_facilities']],
        hovertemplate='<b>Distancia Promedio</b>: %{x:.1f} km<br>' +
                      '<b>Costo Total</b>: $%{y:,.0f}<br>' +
                      '%{text}<extra></extra>'
    ))
    
    fig.update_layout(
        title="Pareto Frontier: Trade-off Costo vs Nivel de Servicio",
        xaxis_title="Distancia Promedio de Servicio (km) - MENOR ES MEJOR",
        yaxis_title="Costo Total ($) - MENOR ES MEJOR",
        width=800,
        height=600,
        annotations=[
            dict(
                x=0.5, y=1.1,
                xref='paper', yref='paper',
                text='← Mejor Servicio | Menor Costo →',
                showarrow=False,
                font=dict(size=12, color='gray')
            )
        ]
    )
    
    fig.show()
    
    # Análisis del trade-off
    print("\n🎯 ANÁLISIS DEL TRADE-OFF")
    print("="*60)
    
    best_cost_idx = df_pareto['cost'].idxmin()
    best_service_idx = df_pareto['avg_distance_km'].idxmin()
    
    print(f"\n💰 MEJOR COSTO:")
    print(f"   Costo: ${df_pareto.loc[best_cost_idx, 'cost']:,.0f}")
    print(f"   Distancia promedio: {df_pareto.loc[best_cost_idx, 'avg_distance_km']:.1f} km")
    print(f"   Facilities: {int(df_pareto.loc[best_cost_idx, 'num_facilities'])}")
    
    print(f"\n🚀 MEJOR SERVICIO:")
    print(f"   Costo: ${df_pareto.loc[best_service_idx, 'cost']:,.0f}")
    print(f"   Distancia promedio: {df_pareto.loc[best_service_idx, 'avg_distance_km']:.1f} km")
    print(f"   Facilities: {int(df_pareto.loc[best_service_idx, 'num_facilities'])}")
    
    cost_increase = (df_pareto.loc[best_service_idx, 'cost'] - df_pareto.loc[best_cost_idx, 'cost']) / df_pareto.loc[best_cost_idx, 'cost'] * 100
    distance_reduction = (df_pareto.loc[best_cost_idx, 'avg_distance_km'] - df_pareto.loc[best_service_idx, 'avg_distance_km']) / df_pareto.loc[best_cost_idx, 'avg_distance_km'] * 100
    
    print(f"\n📊 TRADE-OFF:")
    print(f"   Incremento de costo: +{cost_increase:.1f}%")
    print(f"   Reducción de distancia: -{distance_reduction:.1f}%")
    print(f"   Ratio: {distance_reduction / cost_increase:.2f} (reducción distancia por % de costo)")
    
    # Guardar resultados
    output_file = OUTPUT_DIR / "pareto_frontier.csv"
    df_pareto.to_csv(output_file, index=False)
    print(f"\n💾 Pareto frontier guardado: {output_file}")
else:
    print("⚠️ No hay datos de Pareto para visualizar")

**Interpretación del Pareto Frontier:**

El gráfico muestra:
- **Eje X**: Distancia promedio de servicio (km) - MENOR es MEJOR
- **Eje Y**: Costo total ($) - MENOR es MEJOR
- **Color**: Número de facilities abiertas

**¿Cómo elegir la solución final?**

Depende de la estrategia:
- **Liderazgo en costos**: Punto con menor costo (más a la derecha)
- **Diferenciación por servicio**: Punto con menor distancia (más arriba)
- **Balance**: Punto en el "codo" de la curva (mejor ratio costo/servicio)

**Ejemplo de decisión:**
```
Opción A: $500k, distancia 150 km, 3 DCs
Opción B: $650k, distancia 80 km, 5 DCs
→ ¿Vale la pena +$150k (+30%) para -70 km (-47%)?
```

Esta decisión involucra a Finance, Operations y Strategy.

## 9️⃣ Mapa de Red Logística

In [ ]:
if ORTOOLS_AVAILABLE and status == pywraplp.Solver.OPTIMAL:
    # Preparar datos para mapa
    open_facilities_df = df_facilities[df_facilities['location_id'].isin(open_facilities)].copy()
    open_facilities_df['type'] = 'Facility (Abierta)'
    
    closed_facilities_df = df_facilities[~df_facilities['location_id'].isin(open_facilities)].copy()
    closed_facilities_df['type'] = 'Facility (Cerrada)'
    
    customer_locations_map = customer_locations.copy()
    customer_locations_map['type'] = 'Customer'
    
    # Combinar
    map_data = pd.concat([
        open_facilities_df[['latitude', 'longitude', 'location_id', 'type']],
        closed_facilities_df[['latitude', 'longitude', 'location_id', 'type']],
        customer_locations_map[['latitude', 'longitude', 'location_id', 'type']].head(50)  # Limitar customers para claridad
    ])
    
    # Mapa
    fig = px.scatter_mapbox(
        map_data,
        lat='latitude',
        lon='longitude',
        color='type',
        size=[15 if t == 'Facility (Abierta)' else 5 for t in map_data['type']],
        hover_name='location_id',
        title="Red Logística Optimizada",
        color_discrete_map={
            'Facility (Abierta)': 'green',
            'Facility (Cerrada)': 'red',
            'Customer': 'blue'
        },
        zoom=3,
        height=600
    )
    
    fig.update_layout(mapbox_style="open-street-map")
    fig.show()
    
    print(f"\n🗺️ Mapa generado con:")
    print(f"   - Facilities abiertas: {len(open_facilities)} (verde)")
    print(f"   - Facilities cerradas: {len(closed_facilities_df)} (rojo)")
    print(f"   - Customers mostrados: 50 (azul)")
else:
    print("⚠️ No hay solución para visualizar mapa")

**Mapa de red optimizada:**

El mapa visualiza:
- 🟢 **Facilities abiertas** (verde, grande): DCs que permanecen operativos
- 🔴 **Facilities cerradas** (rojo, pequeño): DCs que se cierran para ahorrar costos
- 🔵 **Customers** (azul, pequeño): Puntos de demanda (mostramos muestra de 50)

**Insights del mapa:**
- Concentración geográfica de facilities abiertas
- Áreas con baja cobertura (distancias largas)
- Oportunidades de consolidación regional

En producción, agregaríamos:
- Líneas de asignación facility→customer
- Tamaño proporcional a volumen/demanda
- Filtros interactivos por región/producto

## 🎓 Conclusiones

**Aprendizajes Clave:**
1. ✅ **Optimización Multiobjetivo**: Trade-off explícito entre costo y servicio
2. ✅ **Facility Location Problem**: Decisiones estratégicas de red logística
3. ✅ **Pareto Frontier**: No existe solución única "óptima" - depende de prioridades de negocio
4. ✅ **Google OR-Tools**: Solver de alta performance para MIP (Mixed Integer Programming)

**Resultados Típicos:**
- **Reducción de costos**: 15-25% vs configuración inicial
- **Mejora en servicio**: -20% en distancia promedio
- **Consolidación**: Menos facilities con mayor utilización (economías de escala)

**Elementos del Modelo:**
```
Minimize: Σ(fixed_cost[f] * y[f]) + Σ(transport_cost[f,c] * demand[c] * x[f,c])

Subject to:
  1. Σ(x[f,c]) = 1  ∀c                    (cada customer servido 100%)
  2. Σ(x[f,c] * demand[c]) ≤ capacity[f]  (capacidad facilities)
  3. x[f,c] ≤ y[f]                        (solo asignar si facility abierta)
  4. Σ(y[f]) ≤ MAX_FACILITIES             (restricción presupuestaria)
  5. Avg_Distance ≤ SERVICE_LEVEL         (restricción de servicio)

Variables:
  - y[f] ∈ {0,1}     (binaria: abrir facility)
  - x[f,c] ∈ [0,1]   (continua: fracción asignada)
```

**Extensiones del Modelo:**
- **Multi-echelon**: Suppliers → DCs → Stores (3 niveles)
- **Time-phased**: Capacidades variables por período
- **Modular capacities**: Abrir/cerrar líneas incrementales
- **Stochastic demand**: Optimización robusta con incertidumbre
- **CO2 emissions**: Añadir objetivo ambiental

**Pareto Frontier - Interpretación:**
- Cada punto en la frontera es **eficiente de Pareto** (no dominado)
- Moverse hacia mejor servicio **siempre** incrementa costo
- La pendiente indica **marginal cost of service improvement**
- Decisión final depende de:
  - Estrategia competitiva (liderazgo en costos vs diferenciación)
  - SLA comprometidos con clientes
  - Restricciones presupuestarias

**Casos de Uso en Supply Chain:**
- Diseño de red de distribución (DC location)
- Sourcing strategy (supplier selection)
- Warehouse consolidation projects
- Network redesign post-merger
- Capacity expansion planning

**Implementación en Producción:**
1. **Data Collection**: EDI/ERP para demanda, TMS para costos transporte
2. **Model Calibration**: Validar con datos históricos
3. **Scenario Analysis**: What-if con cambios de demanda/costos
4. **Change Management**: Plan de transición de red actual → optimizada
5. **Continuous Improvement**: Re-optimizar anualmente

**Métricas de Éxito:**
- **Cost to Serve**: $/unidad entregada
- **Service Level**: % entregas en <24h / <48h
- **Utilization**: % capacidad usada en facilities
- **Flexibility**: Capacidad de absorber demand spikes

**Herramientas Complementarias:**
- **Gurobi/CPLEX**: Solvers comerciales (más rápidos para problemas grandes)
- **PuLP**: Alternativa Python más simple (usada en OR-02, OR-08)
- **Supply Chain Guru**: Software especializado (LLamasoft/Coupa)

---

**🔗 Notebooks Relacionados:**
- [OR-04: Multi-Echelon Inventory](../50_optimization_or/OR-04-multi_echelon_inventory.ipynb)
- [OR-08: Production Scheduling](../50_optimization_or/OR-08-production_scheduling.ipynb)
- [DS-03: Service Level-Cost Tradeoff](../30_data_science_ml/DS-03-service_level_cost_tradeoff.ipynb)
- [BA-02: Cost to Serve](../40_business_analytics_bi/BA-02-cost_to_serve.ipynb)

## 📊 Resumen Ejecutivo

**Lo que logramos:**
- ✅ Modelo MIP con variables binarias (abrir DCs) y continuas (asignaciones)
- ✅ Optimización de costos totales (fijos + transporte)
- ✅ Pareto Frontier explorando trade-off costo vs servicio
- ✅ Análisis de utilización de facilities y nivel de servicio
- ✅ Visualización geográfica de red optimizada

**Resultados típicos:**
- Reducción de costos: 15-25% vs configuración actual
- Consolidación: De 8+ DCs a 3-5 óptimos
- Utilización: 75-85% (evitando sobrecarga o capacidad ociosa)
- Nivel de servicio: Variable según punto elegido en Pareto

**Decisiones habilitadas:**
- Rediseño estratégico de red nacional/regional
- Trade-off explícito: ¿Vale la pena +$X para -Y km de distancia?
- Cierre/apertura de facilities con impacto cuantificado
- Negociación con stakeholders basada en datos (no intuición)

## 🛠️ Funciones Reutilizables

In [ ]:
def export_network_to_geojson(open_facilities, assignments_df, facilities_df, customers_df, output_path: Path):
    """
    Exportar red logística a formato GeoJSON para visualización externa.
    
    Args:
        open_facilities: Lista de facility IDs abiertas
        assignments_df: DataFrame con asignaciones facility-customer
        facilities_df: DataFrame con datos de facilities
        customers_df: DataFrame con datos de customers
        output_path: Directorio de salida
    """
    import json
    
    geojson = {
        "type": "FeatureCollection",
        "features": []
    }
    
    # Facilities
    for _, facility in facilities_df[facilities_df['location_id'].isin(open_facilities)].iterrows():
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [facility['longitude'], facility['latitude']]
            },
            "properties": {
                "type": "facility",
                "id": facility['location_id'],
                "capacity": int(facility['capacity'])
            }
        }
        geojson["features"].append(feature)
    
    # Assignments (lineas)
    for _, assignment in assignments_df.iterrows():
        facility_row = facilities_df[facilities_df['location_id'] == assignment['facility']].iloc[0]
        customer_row = customers_df[customers_df['location_id'] == assignment['customer']].iloc[0]
        
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "LineString",
                "coordinates": [
                    [facility_row['longitude'], facility_row['latitude']],
                    [customer_row['longitude'], customer_row['latitude']]
                ]
            },
            "properties": {
                "type": "assignment",
                "facility": assignment['facility'],
                "customer": assignment['customer'],
                "demand": float(assignment['demand_served'])
            }
        }
        geojson["features"].append(feature)
    
    # Guardar
    output_file = output_path / "network_map.geojson"
    with open(output_file, 'w') as f:
        json.dump(geojson, f, indent=2)
    
    print(f"💾 GeoJSON exportado: {output_file}")
    print(f"   Puede visualizarse en: https://geojson.io/")

# Ejemplo de uso:
# export_network_to_geojson(open_facilities, df_assignments, df_facilities, customer_locations, OUTPUT_DIR)

## 📝 Notas de Operación (Costes, Retención, Gobernanza)

**Costes**
- Consideraciones de almacenamiento/cómputo/visualización.

**Retención**
- Política por zonas (raw/curated/analytics) y ventanas temporales.

**Gobernanza**
- Calidad de datos, seguridad/PII, linaje, versionado de modelos/artefactos.